# MMseqs2 Tool Examples

This notebook demonstrates three MMseqs2 tools:
1. **`run_mmseqs_search_proteins`** — Search protein sequences against a database
2. **`run_mmseqs_clustering`** — Cluster sequences by similarity to reduce redundancy
3. **`run_mmseqs_search_genomes`** — Nucleotide genome-to-genome search

MMseqs2 binaries are automatically downloaded on first use via `EnvManager`.

## Input/Output Schema

### `mmseqs_clustering`

#### Input (`MmseqsClusteringInput`)
| Field | Type | Description |
|-------|------|-------------|
| input_sequences | List[str] | List of sequences to cluster |
| sequence_ids | Optional[List[str]] | Optional sequence identifiers (defaults to seq_0, seq_1, ...) |

#### Config (`MmseqsClusteringConfig`)
| Field | Type | Default | Description |
|-------|------|---------|-------------|
| min_seq_id | float | 0.60 | Minimum sequence identity threshold for clustering (0.0-1.0) |

#### Output (`MmseqsClusteringOutput`)
| Field | Type | Description |
|-------|------|-------------|
| results | List[MmseqsClusterResult] | List of clustering results, one per input sequence |

Each `MmseqsClusterResult` contains:
| Field | Type | Description |
|-------|------|-------------|
| sequence_id | str | Input sequence identifier |
| input_sequence | str | The original input sequence |
| cluster_id | str | Cluster identifier (representative sequence ID) |
| is_representative | bool | Whether this sequence is the cluster representative |

---

### `mmseqs_search_proteins`

#### Input (`MmseqsSearchProteinsInput`)
| Field | Type | Description |
|-------|------|-------------|
| query_sequences | List[str] | List of protein sequences to search |
| sequence_ids | Optional[List[str]] | Optional sequence identifiers (defaults to seq_0, seq_1, ...) |
| mmseqs_db | str | Path to target database (FASTA file or MMseqs2 database) |

#### Config (`MmseqsSearchProteinsConfig`)
| Field | Type | Default | Description |
|-------|------|---------|-------------|
| threads | int | 96 | Number of CPU threads for parallel processing |
| split | int | 0 | Memory management mode (0=auto, higher uses less memory) |
| sensitivity | float | 4.0 | Search sensitivity (1.0=fast, 7.5=very sensitive) |
| only_top_hits | bool | True | If True, keep only the best hit per query sequence |

#### Output (`MmseqsSearchProteinsOutput`)
| Field | Type | Description |
|-------|------|-------------|
| results | List[MmseqsSequenceSearchResult] | List of search results, one per input sequence |

Each `MmseqsSequenceSearchResult` contains:
| Field | Type | Description |
|-------|------|-------------|
| query_id | str | Query sequence identifier |
| query_sequence | str | The input query sequence |
| hits | List[MmseqsHit] | List of hits for this query |

Each `MmseqsHit` contains:
| Field | Type | Description |
|-------|------|-------------|
| target_id | str | Target sequence identifier from the database |
| pident | float | Percentage identity (0-100) |
| evalue | float | E-value (expected number of chance matches) |

---

### `mmseqs_search_genomes`

#### Input (`MmseqsSearchGenomesInput`)
| Field | Type | Description |
|-------|------|-------------|
| query_genomes | List[str] | List of query genome sequences (nucleotide) |
| query_ids | Optional[List[str]] | Optional query identifiers (defaults to seq_0, seq_1, ...) |
| target_genomes | List[str] | List of target genome sequences to search against |
| target_ids | Optional[List[str]] | Optional target identifiers (defaults to target_0, target_1, ...) |

#### Config (`MmseqsSearchGenomesConfig`)
| Field | Type | Default | Description |
|-------|------|---------|-------------|
| search_type | int | 3 | MMseqs2 search type (3=nucleotide vs nucleotide) |
| threads | int | 96 | Number of CPU threads for parallel processing |
| sensitivity | float | 7.5 | Search sensitivity (7.5=very sensitive, default for genomes) |

#### Output (`MmseqsSearchGenomesOutput`)
| Field | Type | Description |
|-------|------|-------------|
| results | List[MmseqsSequenceSearchResult] | List of search results, one per input query genome |

In [1]:
from bio_programming_tools.tools.gene_annotation.mmseqs import (
    run_mmseqs_search_proteins, MmseqsSearchProteinsInput, MmseqsSearchProteinsConfig,
    run_mmseqs_clustering, MmseqsClusteringInput, MmseqsClusteringConfig,
    run_mmseqs_search_genomes, MmseqsSearchGenomesInput, MmseqsSearchGenomesConfig,
)
from pathlib import Path
import tempfile

## 1. Sequence Clustering

Cluster a set of protein sequences to reduce redundancy. Sequences with >=90% identity
will be grouped into the same cluster. This is useful for removing near-duplicates
before running expensive downstream analyses.

In [2]:
# GFP and variants: two near-identical GFP sequences and one distinct myoglobin
sequences = [
    "MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTFSYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDELYK",
    "MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTFSYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDE",
    "MGLSDGEWQLVLNVWGKVEADIPGHGQEVLIRLFKGHPETLEKFDKFKHLKSEDEMKASEDLKKHGATVLTALGGILKKKGHHEAEIKPLAQSHATKHKIPVKYLEFISECIIQVLQSKHPGDFGADAQGAMNKALELFRKDMASNYKELGFQG",
]

cluster_input = MmseqsClusteringInput(input_sequences=sequences)
cluster_config = MmseqsClusteringConfig(min_seq_id=0.90)

result = run_mmseqs_clustering(cluster_input, cluster_config)
print(f"Found {result.num_clusters} clusters from {len(sequences)} sequences")
for r in result:
    print(f"  {r.sequence_id}: cluster={r.cluster_id}, representative={r.is_representative}")

Setting up MMseqs2 standalone environment...
Installing uv package manager...
  Using cached uv-0.10.0-py3-none-macosx_11_0_arm64.whl.metadata (11 kB)
Using cached uv-0.10.0-py3-none-macosx_11_0_arm64.whl (19.8 MB)
Installing Python dependencies...
Installing mmseqs for Darwin/arm64...
  Downloading: 100%|██████████| 20.7M/20.7M [00:01<00:00, 16.3MB/s]
  Download complete (19.8 MB)
  Extracting binaries to /Users/danielguo/Research/darwin/bio-programming/.venvs/mmseqs_env/bin...
  Installed: mmseqs
mmseqs installation complete!
MMseqs2 setup complete!
Found 2 clusters from 3 sequences
  seq_0: cluster=seq_1, representative=False
  seq_1: cluster=seq_1, representative=True
  seq_2: cluster=seq_2, representative=True


## 2. Protein Search

Search protein sequences against a local database. First, we create a small
FASTA database, then search against it.

In [3]:
# Create a small protein database as a FASTA file
tmp_dir = tempfile.mkdtemp(prefix="mmseqs_example_")
db_fasta = Path(tmp_dir) / "proteins.fasta"
db_fasta.write_text("""\
>sp|P69905|HBA_HUMAN Hemoglobin subunit alpha
MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH
GSAQVKGHGKKVADALTNAVAHVDDMPNALSALSDLHAHKLRVDPVNFKL
LSHCLLVTLAAHLPAEFTPAVHASLDKFLASVSTVLTSKYR
>sp|P68871|HBB_HUMAN Hemoglobin subunit beta
MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLST
PDAVMGNPKVKAHGKKVLGAFSDGLAHLDNLKGTFATLSELHCDKLHVDP
ENFRLLGNVLVCVLAHHFGKEFTPPVQAAYQKVVAGVANALAHKYH
>sp|P02144|MYG_HUMAN Myoglobin
MGLSDGEWQLVLNVWGKVEADIPGHGQEVLIRLFKGHPETLEKFDKFKHL
KSEDEMKASEDLKKHGATVLTALGGILKKKGHHEAEIKPLAQSHATKHKIP
VKYLEFISECIIQVLQSKHPGDFGADAQGAMNKALELFRKDMASNYKELGFQG
""")
print(f"Database FASTA written to: {db_fasta}")

# Search hemoglobin beta against the database
query_seq = "MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPKVKAHGKKVLGAFSDGLAHLDNLKGTFATLSELHCDKLHVDPENFRLLGNVLVCVLAHHFGKEFTPPVQAAYQKVVAGVANALAHKYH"

search_input = MmseqsSearchProteinsInput(
    query_sequences=[query_seq],
    mmseqs_db=str(db_fasta),
)
search_config = MmseqsSearchProteinsConfig(
    sensitivity=7.5,
    threads=2,
    only_top_hits=False,
)

result = run_mmseqs_search_proteins(search_input, search_config)
print(f"\nFound {result.total_hits} total hits")
for r in result:
    print(f"  Query {r.query_id}: {r.num_hits} hits")
    for hit in r.hits:
        print(f"    -> {hit.target_id}: {hit.pident:.1f}% identity (evalue={hit.evalue:.2e})")

Database FASTA written to: /var/folders/rs/6dqw0_k1125fl7f7_9h85hgh0000gn/T/mmseqs_example_3o4vfei0/proteins.fasta

Found 2 total hits
  Query seq_0: 2 hits
    -> P68871: 100.0% identity (evalue=4.65e-109)
    -> P69905: 43.4% identity (evalue=1.57e-35)


## 3. Genome Search

Search nucleotide sequences against each other. This uses the full MMseqs2
workflow (createdb, createindex, search, convertalis) for nucleotide-vs-nucleotide
comparisons.

In [4]:
# Human hemoglobin alpha and beta coding sequences (partial)
query_genomes = [
    "ATGGTGCTGTCTCCTGCCGACAAGACCAACGTCAAGGCCGCCTGGGGTAAGGTCGGCGCGCACGCTGGCGAGTATGGTGCGGAGGCCCTGGAGAGGATGTTCCTGTCCTTCCCCACCACCAAGACCTACTTCCCGCACTTCGACCTGAGCCACGGCTCTGCCCAGGTTAAGGGCCACGGCAAGAAGGTGGCCGACGCGCTGACCAACGCCGTGGCGCACGTGGACGACATGCCCAACGCGCTGTCCGCCCTGAGCGACCTGCACGCGCACAAGCTTCGGGTGGACCCGGTCAACTTCAAGCTCCTAAGCCACTGCCTGCTGGTGACCCTGGCCGCCCACCTCCCCGCCGAGTTCACCCCGACCGTGGACGCCGACCTGGCCAGCGTGAGCACCGTGCTGACCTCCAAATACCGTTAA",
]
target_genomes = [
    "ATGGTGCTGTCTCCTGCCGACAAGACCAACGTCAAGGCCGCCTGGGGTAAGGTCGGCGCGCACGCTGGCGAGTATGGTGCGGAGGCCCTGGAGAGGATGTTCCTGTCCTTCCCCACCACCAAGACCTACTTCCCGCACTTCGACCTGAGCCACGGCTCTGCCCAGGTTAAGGGCCACGGCAAGAAGGTGGCCGACGCGCTGACCAACGCCGTGGCGCACGTGGACGACATGCCCAACGCGCTGTCCGCCCTGAGCGACCTGCACGCGCACAAGCTTCGGGTGGACCCGGTCAACTTCAAGCTCCTAAGCCACTGCCTGCTGGTGACCCTGGCCGCCCACCTCCCCGCCGAGTTCACCCCGACCGTGGACGCCGACCTGGCCAGCGTGAGCACCGTGCTGACCTCCAAATACCGTTAA",
    "ATGGTGCACCTGACTCCTGAGGAGAAGTCTGCCGTTACTGCCCTGTGGGGCAAGGTGAACGTGGATGAAGTTGGTGGTGAGGCCCTGGGCAGGCTGCTGGTGGTCTACCCTTGGACCCAGAGGTTCTTTGAGTCCTTTGGGGATCTGTCCACTCCTGATGCTGTTATGGGCAACCCTAAGGTGAAGGCTCATGGCAAGAAAGTGCTCGGTGCCTTTAGTGATGGCCTGGCTCACCTGGACAACCTCAAGGGCACCTTTGCTACACTGAGTGAGCTGCACTGTGACAAGCTGCACGTGGATCCTGAGAACTTCAGGCTCCTGGGCAACGTGCTGGTCTGTGTGCTGGCCCATCACTTTGGCAAAGAATTCACCCCACCAGTGCAGGCTGCCTATCAGAAAGTGGTGGCTGGTGTGGCTAATGCCCTGGCCCACAAGTATCACTAA",
]

genome_input = MmseqsSearchGenomesInput(
    query_genomes=query_genomes,
    target_genomes=target_genomes,
)
genome_config = MmseqsSearchGenomesConfig(threads=2)

result = run_mmseqs_search_genomes(genome_input, genome_config)
print(f"Found {result.total_hits} total hits")
for r in result:
    print(f"  Query {r.query_id}: {r.num_hits} hits")
    for hit in r.hits:
        print(f"    -> {hit.target_id}: {hit.pident:.1f}% identity")

Found 1 total hits
  Query seq_0: 1 hits
    -> seq_0: 100.0% identity


## 4. Export Results

Export all results to an `example_output/` directory.

In [ ]:
output_dir = Path("./example_output")

# Export clustering results to CSV
result.export("mmseqs_clusters", export_path=output_dir, file_format="csv")
print("Clustering results exported to example_output/")